# Train detector toàn biển: YOLO26-OBB

Ảnh toàn cảnh → biển số → crop xoay → OCR. Train **nano** trên GPU. Kernel `.venv-ocr`. Nano dùng ảnh 640, batch 4 trên GPU 4 GB. Không train lại OCR.

## 1. Môi trường

Đã cài trên máy hiện tại. Máy mới: `.venv-ocr\Scripts\python.exe -m pip install -r requirements/detection.txt`.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/detection/yolo26_obb.json").is_file())
os.chdir(ROOT)
if Path(sys.executable).resolve() != (ROOT / ".venv-ocr/Scripts/python.exe").resolve():
    raise RuntimeError("Chọn kernel .venv-ocr rồi Restart Kernel")
from ocr.notebook import run
from plate_detection.data import load_config
VARIANTS = list(load_config()["variants"])
print("YOLO26-OBB:", VARIANTS)
import torch
from plate_detection.train import ultralytics_api
ultralytics_api(load_config())
if not torch.cuda.is_available():
    raise RuntimeError("Thiếu GPU CUDA")
print(torch.__version__, torch.cuda.get_device_name(0))


## 2. Rà nhãn nguồn

Nguồn đã tải: `Vietnamese-License-Plate-1/`. YOLO26 export này chứa polygon. Chuyển polygon thật sang chữ nhật xoay nhỏ nhất; không suy góc từ nhãn AABB. Ảnh train có polygon hỏng/suy biến được cách ly nguyên ảnh trong bản dựng, nguồn vẫn giữ nguyên.

In [ ]:
import json
from plate_detection.data import audit_dataset
from ocr.config import save_json
cfg = load_config()
from plate_detection.quality import apply_split_review
report = apply_split_review(audit_dataset(cfg["raw_dataset"]))
save_json(ROOT / "runs/plate_detection/label_audit.json", report)
print(json.dumps({k:v for k,v in report.items() if k != "records"}, ensure_ascii=False, indent=2))
if not report["ready"]:
    raise ValueError("Dữ liệu chưa đạt; xem errors trước khi train")


## 3. Xem polygon và OBB

Cam: polygon nguồn. Xanh: chữ nhật xoay. OBB không có bốn góc độc lập để bám sát hình thang do phối cảnh.

In [ ]:
from plate_detection.preview import show_labels
show_labels(report, count=12, split="train");


## 4. Dựng dữ liệu OBB

Gộp biển dài/ngắn thành một lớp `license-plate`. Đệm ảnh nếu chữ nhật vượt mép, không cắt khung. Dùng lại bản dựng có cùng nguồn; không ghi đè dữ liệu khác.

Bản hiện tại: **train 3.591 / val 515 / test 258**, 5.252 hộp. Đã cách ly ba ảnh train nhãn hỏng và 24 ảnh trùng nhóm qua split sau khi đối chiếu ảnh. Chưa chứng nhận mọi danh tính khác tên/góc chụp. `normalization_check.json` phải đạt trước khi train.

In [ ]:
from plate_detection.data import prepare_dataset, check_prepared
prepared = Path(cfg["dataset"])
if not prepared.exists():
    print(prepare_dataset(cfg))
else:
    source = json.loads((prepared / "source_audit.json").read_text(encoding="utf-8"))
    expected = [(r["image_sha256"],r["label_sha256"]) for r in report["records"]]
    previous = [(r["image_sha256"],r["label_sha256"]) for r in source["records"]]
    if expected != previous:
        raise ValueError("Nguồn thay đổi; cần cấu hình thư mục dựng mới")
    print("Dùng lại:", prepared / "data.yaml")

print(check_prepared(cfg))


## 5. Train/val YOLO26n-OBB

Đổi `RUN_TRAINING = True` khi bắt đầu. Mỗi bản lưu `best.pt`, `last.pt`, log và đường cong riêng. Chạy lại tiếp tục `last`; bản hoàn tất được bỏ qua. Best chọn theo mAP trên val, không chọn theo test. Nếu có lỗi, xem log trước khi tiếp tục.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    for variant in VARIANTS:
        run(sys.executable, "-m", "plate_detection.train", "train", "--variant", variant)
else:
    print("Chưa train. Đặt RUN_TRAINING = True để train nano.")


## 6. Kiểm val và xuất ONNX FP32

Model nano được lưu trong `runs/plate_detection/yolo26n_obb/onnx/model.onnx`. Xuất best, đối chiếu PT/ONNX trên val. Chỉ chuyển sang test khi có `verified.json`.

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    for variant in VARIANTS:
        run(sys.executable, "-m", "plate_detection.train", "val", "--variant", variant)
        run(sys.executable, "-m", "plate_detection.train", "export", "--variant", variant)
else:
    print("Chưa xuất model.")


## 7. Test detection sau khi chốt model

Kết quả nằm trong `runs/plate_detection/yolo26n_obb/detection_result.json` và hiển thị tại bảng cuối notebook.
ACC/CER toàn luồng nằm ở notebook 05. Giữ `RUN_DETECTION_TEST = False` khi chỉ muốn xem kết quả đã có.

In [ ]:
RUN_DETECTION_TEST = False
if RUN_DETECTION_TEST:
    for variant in VARIANTS:
        run(sys.executable, "-m", "plate_detection.evaluate", "test", "--variant", variant)


## 8. Kết quả YOLO26n-OBB

Bảng dưới là kết quả **test riêng detection**, chưa ghép OCR. Chạy lại cell chỉ đọc và kiểm tra số đo đã lưu, không train hoặc đo lại.

Test 258 ảnh · ONNX FP32 · CPU 4 luồng · warm-up 20 · đo 5 lượt.
mAP dùng evaluator OBB gốc. Precision/Recall ở confidence 0,25 và polygon IoU 0,5.
Speed gồm letterbox + YOLO + giải mã; loại thời gian đọc ảnh và tải model.

Hiện chỉ dùng nano nên bảng có một dòng; Rank = 1 và Speedup = 1×.

In [ ]:
from IPython.display import Markdown, display
from ocr.optimized_benchmark import display_table, POINTER
if POINTER.is_file():
    display(Markdown(display_table("detector")))
else:
    from plate_detection.evaluate import comparison
    display(Markdown(comparison()[1]))


| Model | mAP50 (%) | mAP50-95 (%) | Precision (%) | Recall (%) | Speed ± SD (ms/ảnh) | FPS |
|---|---:|---:|---:|---:|---:|---:|
| YOLO26n-OBB_FP32 | 98.81 | 96.11 | 94.92 | 97.22 | 50.44 ± 8.64 | 19.83 |

Run: D:\DUT\NCKH\dataset\runs\int8_optimization\benchmark\20261113_233439
